# PhiState-MoE: add code + general knowledge, then continue to 60K

This creates a **separate expanded corpus** from the original prepared corpus, keeps the existing 8,192-token BPE tokenizer and original validation/test bins, and continues from a verified 50K optimizer checkpoint. It adds up to 4M code tokens and 4M general-knowledge tokens, with a document-level 5% validation split for the new examples. The added-data source/license details are recorded in `provenance.json`.

The original corpus and checkpoint are never overwritten. The new run is saved in Google Drive and writes recovery checkpoints every 500 steps. If Colab disconnects, run this notebook again; it resumes from the last saved expanded-corpus checkpoint. Keep the runtime connected during a segment.

The code data is a small MIT-declared Python instruction/output dataset; general text is English Wikipedia, whose source material carries CC BY-SA/GFDL attribution terms. This is a modest extension, not a guarantee of better code correctness or general knowledge.


In [3]:
from pathlib import Path
import json, shutil, subprocess, sys, importlib.util
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/phistate_l4')
REPO = ROOT / 'quantoniumoslinux'
LAB = REPO / 'runtime/experiments/phi_runtime_lab'
DRIVE = Path('/content/drive/MyDrive/Quantonium/PhiState-L4')
RUN_DRIVE = DRIVE / '50k'
BASE_DATA = DRIVE / 'data/hf_coding_knowledge_v2'
EXPANDED_DATA = DRIVE / 'data/hf_coding_knowledge_v2_expanded'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Anharmoniclabs/quantoniumoslinux.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
if importlib.util.find_spec('datasets') is None or importlib.util.find_spec('sentencepiece') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'sentencepiece'], check=True)
assert (LAB / 'moe_data_extend.py').is_file() and (LAB / 'pretrain_continue_expanded.py').is_file(), 'Pull the updated main branch before continuing.'
assert BASE_DATA.joinpath('manifest.json').is_file(), f'Original corpus not found: {BASE_DATA}'
print('Repository:', REPO)
print('Original corpus:', BASE_DATA)
print('Expanded corpus will be:', EXPANDED_DATA)


Mounted at /content/drive
Repository: /content/phistate_l4/quantoniumoslinux
Original corpus: /content/drive/MyDrive/Quantonium/PhiState-L4/data/hf_coding_knowledge_v2
Expanded corpus will be: /content/drive/MyDrive/Quantonium/PhiState-L4/data/hf_coding_knowledge_v2_expanded


In [4]:
import torch

progress_path = RUN_DRIVE / 'progress.json'
assert progress_path.is_file(), 'Missing 50K progress.json in Google Drive.'
progress = json.loads(progress_path.read_text())
assert progress.get('step', 0) >= 50000, f'Requires the completed 50K checkpoint; current saved step is {progress.get("step")}'
slot = progress['active_slot']
CHECKPOINT = RUN_DRIVE / f'slot_{slot}.last.pt'
BEST = RUN_DRIVE / f'slot_{slot}.pt'
if not CHECKPOINT.is_file() or not BEST.is_file():
    CHECKPOINT = RUN_DRIVE / 'milestones/phi_50000.last.pt'
    BEST = RUN_DRIVE / 'milestones/phi_50000.pt'
assert CHECKPOINT.is_file() and BEST.is_file(), 'Verified 50K best/latest checkpoint pair not found.'
ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
assert ck['step'] >= 50000 and ck['data_fingerprint'] == progress['data_fingerprint']
assert 'optimizer_state' in ck, 'Optimizer state is required for exact continuation.'
print('Verified source step:', ck['step'])
print('Verified source fingerprint:', ck['data_fingerprint'])
print('Source checkpoint remains untouched:', CHECKPOINT)
del ck

# If an expanded run already exists, resume from its latest checkpoint.
OUT_BEST = RUN_DRIVE / 'phi_60000.pt'
OUT_LAST = RUN_DRIVE / 'phi_60000.last.pt'
if OUT_LAST.is_file():
    SOURCE_FOR_TRAINING = OUT_LAST
else:
    SOURCE_FOR_TRAINING = CHECKPOINT
print('Continuation starts from:', SOURCE_FOR_TRAINING)


Verified source step: 50000
Verified source fingerprint: 29e4997527c221f48180d30f5d5d3245d0b5476e69a3b735d5b0895204982474
Source checkpoint remains untouched: /content/drive/MyDrive/Quantonium/PhiState-L4/50k/slot_b.last.pt
Continuation starts from: /content/drive/MyDrive/Quantonium/PhiState-L4/50k/phi_60000.last.pt


In [5]:
if not EXPANDED_DATA.joinpath('manifest.json').is_file():
    cmd = [sys.executable, '-u', str(LAB / 'moe_data_extend.py'),
           '--base', str(BASE_DATA), '--out', str(EXPANDED_DATA),
           '--max-code-tokens', '4000000', '--max-general-tokens', '4000000',
           '--validation-percent', '5']
    subprocess.run(cmd, check=True)
else:
    manifest = json.loads((EXPANDED_DATA / 'manifest.json').read_text())
    provenance = json.loads((EXPANDED_DATA / 'provenance.json').read_text())
    assert manifest['extension']['base_data_fingerprint'] == json.loads((BASE_DATA / 'manifest.json').read_text())['data_fingerprint']
    print('Reusing persisted expanded corpus:', EXPANDED_DATA)
    print('Added token counts:', {k: provenance['selection']['added'][k]['train'] + provenance['selection']['added'][k]['val'] for k in ('code', 'general')})


Reusing persisted expanded corpus: /content/drive/MyDrive/Quantonium/PhiState-L4/data/hf_coding_knowledge_v2_expanded
Added token counts: {'code': 3924017, 'general': 4000168}


In [6]:
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > L4 GPU and reconnect.'
cmd = [sys.executable, '-u', str(LAB / 'pretrain_continue_expanded.py'),
       '--checkpoint', str(SOURCE_FOR_TRAINING), '--data-dir', str(EXPANDED_DATA),
       '--out', str(OUT_BEST), '--target-steps', '60000', '--eval-every', '500',
       '--eval-batches', '16', '--threads', '2', '--device', 'cuda']
print('CONTINUING: existing trained weights -> 60,000 total steps on L4')
print('Best checkpoint:', OUT_BEST)
print('Resumable latest checkpoint:', OUT_LAST)
subprocess.run(cmd, check=True)


CONTINUING: existing trained weights -> 60,000 total steps on L4
Best checkpoint: /content/drive/MyDrive/Quantonium/PhiState-L4/50k/phi_60000.pt
Resumable latest checkpoint: /content/drive/MyDrive/Quantonium/PhiState-L4/50k/phi_60000.last.pt


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/phistate_l4/quantoniumoslinux/runtime/experiments/phi_runtime_lab/pretrain_continue_expanded.py', '--checkpoint', '/content/drive/MyDrive/Quantonium/PhiState-L4/50k/phi_60000.last.pt', '--data-dir', '/content/drive/MyDrive/Quantonium/PhiState-L4/data/hf_coding_knowledge_v2_expanded', '--out', '/content/drive/MyDrive/Quantonium/PhiState-L4/50k/phi_60000.pt', '--target-steps', '60000', '--eval-every', '500', '--eval-batches', '16', '--threads', '2', '--device', 'cuda'], returncode=0)

In [7]:
if OUT_LAST.is_file():
    latest = torch.load(OUT_LAST, map_location='cpu', weights_only=True)
    best = torch.load(OUT_BEST, map_location='cpu', weights_only=True) if OUT_BEST.is_file() else {}
    print('SAVED STEP:', latest['step'], '/ 60000')
    print('NEW-VALIDATION BEST BPT:', best.get('best_validation_bpt'))
    print('EXPANDED FINGERPRINT:', latest['data_fingerprint'])
    print('Drive recovery checkpoint:', OUT_LAST)
    del latest, best
else:
    print('Training did not save a new checkpoint yet; inspect the error/output above.')


SAVED STEP: 60000 / 60000
NEW-VALIDATION BEST BPT: 6.63308
EXPANDED FINGERPRINT: da26d001a8a80ac6079fd160a56ee9ad2e03cd442eb8f2fab94f0c0ce99bc7ca
Drive recovery checkpoint: /content/drive/MyDrive/Quantonium/PhiState-L4/50k/phi_60000.last.pt


In [8]:
import ast
import sys
import torch
import sentencepiece as spm

sys.path.insert(0, str(LAB))
from moe_train import ckpt_load

FIXED_TOKENIZER = DRIVE / "whitespace_repair_v1" / "tokenizer.model"
assert OUT_LAST.is_file(), f"Missing checkpoint: {OUT_LAST}"
assert FIXED_TOKENIZER.is_file(), f"Missing repaired tokenizer: {FIXED_TOKENIZER}"

tokenizer = spm.SentencePieceProcessor(model_file=str(FIXED_TOKENIZER))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)

model, metadata = ckpt_load(str(OUT_LAST))
assert metadata["step"] == 60000, f"Expected step 60000, got {metadata['step']}"
assert model.cfg.vocab == tokenizer.vocab_size(), "Tokenizer/model vocabulary mismatch"

model = model.to(device).eval()
print("Checkpoint step:", metadata["step"])
print("Generation device:", device)

prompts = [
    ("fibonacci", "code", "def fibonacci(n):\n    "),
    ("add", "code", "def add(a, b):\n    "),
    (
        "ram_vs_storage",
        "general",
        "User: What is the difference between RAM and storage?\nAssistant: ",
    ),
    ("operating_system", "general", "The purpose of an operating system is"),
]

def generate(prompt, seed=20261030, new_tokens=64, temperature=0.8, top_k=32):
    ids = tokenizer.encode(prompt, out_type=int)
    rng = torch.Generator(device=device).manual_seed(seed)
    state = None

    with torch.inference_mode():
        for token_id in ids:
            x = torch.tensor([[token_id]], dtype=torch.long, device=device)
            logits, state = model.step(x, state)

        produced = []
        for _ in range(new_tokens):
            scores = logits[0, -1] / temperature
            values, choices = torch.topk(scores, min(top_k, scores.numel()))
            probs = torch.softmax(values, dim=-1)
            picked = torch.multinomial(probs, 1, generator=rng)
            next_id = int(choices[picked])

            produced.append(next_id)
            x = torch.tensor([[next_id]], dtype=torch.long, device=device)
            logits, state = model.step(x, state)

    return tokenizer.decode(ids + produced)

for name, kind, prompt in prompts:
    output = generate(prompt)
    print(f"\n--- {name} ---")
    print(output)

    if kind == "code":
        try:
            ast.parse(output)
            print("Python syntax parses: True")
        except SyntaxError as error:
            print(f"Python syntax parses: False — {error.msg} at line {error.lineno}")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Checkpoint step: 60000
Generation device: cuda

--- fibonacci ---
def fibonacci(n):
    ^^ ..^||c] Utimization-two-cmd-f-8 - ^[-b|c2-b-bula-svi-ban-bid-fat-wikroca-fro-a-car
Python syntax parses: False — unmatched ']' at line 2

--- add ---
def add(a, b):
    <|code|> /* Copyright utrd sidney. * * © 2017 ^ Ava (c) 2012 bdi, uttarv (b) or dpm) as fontal s = tap.login, n = np.rvn.
Python syntax parses: False — invalid character '©' (U+00A9) at line 2

--- ram_vs_storage ---
User: What is the difference between RAM and storage?
Assistant: ----------jango List: "Coard. Learning a bone "a" is a "larging" or "How" toxico" "Super", "Ton" : "Demo", "Shell" : "Myy

--- operating_system ---
The purpose of an operating system is in the right of the Cathath, in the meltle of these in the Surgent, but that the cerver the middle of the nestern of mouth by the UP's Louthern Ray, I've removen of his own, and


In [9]:
import hashlib
import json

REPAIR_DIR = DRIVE / "whitespace_repair_v1"
marker = REPAIR_DIR / "raw_recovery_complete.json"
assert marker.is_file(), f"Missing recovery marker: {marker}"

recovery = json.loads(marker.read_text(encoding="utf-8"))
RAW_DIR = Path(recovery["raw_directory"])
manifest = json.loads((BASE_DATA / "manifest.json").read_text(encoding="utf-8"))

split_totals = {
    domain: {
        split: {"documents": 0, "bytes": 0}
        for split in ("train", "val", "test")
    }
    for domain in ("code", "general")
}
seen_hashes = {}
duplicate_documents = 0
hash_errors = []

def split_for_sha256(digest):
    # Manifest rule: SHA-256 buckets 0–9=val, 10–19=test, 20–99=train.
    bucket = int(digest, 16) % 100
    if bucket < 10:
        return "val"
    if bucket < 20:
        return "test"
    return "train"

for domain in ("code", "general"):
    path = RAW_DIR / f"{domain}.jsonl"
    assert path.is_file(), f"Missing recovered file: {path}"

    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            row = json.loads(line)
            raw = row["text"].encode("utf-8")
            digest = hashlib.sha256(raw).hexdigest()

            if digest != row.get("sha256"):
                hash_errors.append(f"{domain}.jsonl:{line_number}")
                continue

            split = split_for_sha256(digest)
            split_totals[domain][split]["documents"] += 1
            split_totals[domain][split]["bytes"] += len(raw)

            if digest in seen_hashes:
                duplicate_documents += 1
                if seen_hashes[digest] != split:
                    raise AssertionError(f"Duplicate hash assigned to two splits: {digest}")
            else:
                seen_hashes[digest] = split

print("Raw data directory:", RAW_DIR)
print("Duplicate documents across recovered files:", duplicate_documents)
print("SHA-256 mismatches:", hash_errors[:10] if hash_errors else "None")

all_match = not hash_errors

for domain in ("code", "general"):
    print(f"\n{domain.upper()}")
    for split in ("train", "val", "test"):
        actual = split_totals[domain][split]
        expected = manifest["documents"][domain]
        expected_docs = expected[f"{split}_documents"]
        expected_bytes = expected[f"{split}_bytes"]

        matches = (
            actual["documents"] == expected_docs
            and actual["bytes"] == expected_bytes
        )
        all_match &= matches

        print(
            f"{split}: {actual['documents']} docs / {actual['bytes']} bytes"
            f" | original manifest: {expected_docs} docs / {expected_bytes} bytes"
            f" | totals match: {matches}"
        )

print("\nAll split totals match original manifest:", all_match)
print("No files were modified; no training was started.")

Raw data directory: /content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1/raw_recovery_mvly89y6
Duplicate documents across recovered files: 0
SHA-256 mismatches: None

CODE
train: 7212 docs / 52618784 bytes | original manifest: 7238 docs / 53743483 bytes | totals match: False
val: 884 docs / 6856014 bytes | original manifest: 890 docs / 6876743 bytes | totals match: False
test: 916 docs / 7603868 bytes | original manifest: 884 docs / 6368994 bytes | totals match: False

GENERAL
train: 11553 docs / 53785562 bytes | original manifest: 11572 docs / 53807487 bytes | totals match: False
val: 1457 docs / 6555764 bytes | original manifest: 1419 docs / 6952722 bytes | totals match: False
test: 1365 docs / 6754265 bytes | original manifest: 1384 docs / 6335382 bytes | totals match: False

All split totals match original manifest: False
No files were modified; no training was started.


In [10]:
from pathlib import Path

candidates = [
    LAB / "hf_phistate_data_v2.py",
    LAB / "moe_data_extend.py",
    LAB / "hf_instruction_data.py",
    RAW_DIR / "recovery_builder.py",
]

for path in candidates:
    if not path.is_file():
        continue

    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    hits = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in ("sha256", "split_policy", "bucket"))
    ]

    if hits:
        print(f"\n===== {path.name} =====")
        shown = set()
        for i in hits:
            start, end = max(0, i - 4), min(len(lines), i + 6)
            if any(j in shown for j in range(start, end)):
                continue
            print(f"\n--- lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")
                shown.add(j)


===== hf_phistate_data_v2.py =====

--- lines 64-73 ---
64:                 body = body.replace("\x00", "").strip()
65:                 raw = body.encode("utf-8")
66:                 if len(raw) < 160 or len(raw) > args.max_document_kb * 1024 or SECRET.search(body):
67:                     continue
68:                 digest = hashlib.sha256(raw).hexdigest()
69:                 if digest in seen:
70:                     continue
71:                 if used + len(raw) > cap:
72:                     break
73:                 seen.add(digest)

===== moe_data_extend.py =====

--- lines 22-31 ---
22: GENERAL_ID = "wikimedia/wikipedia"
23: GENERAL_CONFIG = "20231101.en"
24: 
25: 
26: def sha256_file(path: Path) -> str:
27:     digest = hashlib.sha256()
28:     with path.open("rb") as stream:
29:         for block in iter(lambda: stream.read(1024 * 1024), b""):
30:             digest.update(block)
31:     return digest.hexdigest()

--- lines 58-67 ---
58:     train_key, val_key = f"train_{do

In [11]:
from pathlib import Path

terms = (
    "split_policy",
    "train_documents",
    "val_documents",
    "test_documents",
    "train_code",
    "val_code",
    "test_code",
    "sha256",
    "bucket",
)

skip_dirs = {
    ".git", ".venv", "venv", "node_modules",
    "data", "checkpoints", "__pycache__",
}
extensions = {".py", ".ipynb", ".json", ".md"}

roots = [LAB, *list(LAB.parents[:3])]
seen_files = set()
hits_found = 0

for root in roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue
        if any(part in skip_dirs for part in path.parts):
            continue
        if path in seen_files:
            continue
        seen_files.add(path)

        try:
            lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        except OSError:
            continue

        matched = [
            i for i, line in enumerate(lines)
            if any(term in line.lower() for term in terms)
        ]
        if not matched:
            continue

        print(f"\n===== {path} =====")
        shown = set()
        for i in matched:
            start, end = max(0, i - 3), min(len(lines), i + 5)
            if any(j in shown for j in range(start, end)):
                continue
            print(f"\n--- lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")
                shown.add(j)
            hits_found += 1
            if hits_found >= 40:
                break

        if hits_found >= 40:
            break
    if hits_found >= 40:
        break

if hits_found == 0:
    print("No split-assignment code found in the searched project files.")


===== /content/phistate_l4/quantoniumoslinux/runtime/experiments/phi_runtime_lab/lab.py =====

--- lines 86-93 ---
86:     # Exact-content dedupe, useful when datasets include redundant copies.
87:     unique = {}
88:     for name, raw in all_docs:
89:         digest = hashlib.sha256(raw).hexdigest()
90:         if digest not in unique:
91:             unique[digest] = (name, raw)
92:     docs = list(unique.values())
93:     if len(docs) >= 10:

--- lines 113-120 ---
113:     if min(lengths.values()) < 512:
114:         raise RuntimeError(f"Corpus too small for useful training: {lengths}; supply more text files")
115:     metadata = {
116:         "source": str(location), "sha256_fingerprints": [hashlib.sha256(c).hexdigest() for c in chunks.values()],
117:         "file_count": len(docs), "split_method": split, "bytes": lengths,
118:         "warning": "Check for near-duplicates; same project documents may share passages across splits."
119:     }
120:     return {k: to_tensor(v) for 

In [12]:
from pathlib import Path
import json
import subprocess
import sys
import torch

LAB = Path(LAB)
BASE_DATA = Path(BASE_DATA)
EXPANDED_DATA = Path(EXPANDED_DATA)
SOURCE_CHECKPOINT = Path(OUT_LAST)

REPAIR_DIR = Path(DRIVE) / "whitespace_repair_v1"
REPAIRED_DATA = REPAIR_DIR / "corpus_repaired_v1"
REPAIR_RUN = REPAIR_DIR / "run_60k_to_61k"

repair_script = LAB / "repair_whitespace_corpus.py"
continue_script = LAB / "continue_whitespace_repair.py"

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
for path in (repair_script, continue_script, SOURCE_CHECKPOINT):
    assert path.is_file(), f"Missing required file: {path}"

# Verify an existing repair, or build it using exact original-stream matching.
# If verification fails, check=True stops this cell before training.
subprocess.run([
    sys.executable, "-u", str(repair_script),
    "--base-data", str(BASE_DATA),
    "--repair-dir", str(REPAIR_DIR),
    "--out", str(REPAIRED_DATA),
], check=True)

print("\nStarting/resuming the repaired-data run to step 61,000.", flush=True)
print("Checkpoints:", REPAIR_RUN, flush=True)

command = [
    sys.executable, "-u", str(continue_script),
    "--checkpoint", str(SOURCE_CHECKPOINT),
    "--source-data", str(EXPANDED_DATA),
    "--data-dir", str(REPAIRED_DATA),
    "--run-dir", str(REPAIR_RUN),
    "--expected-source-step", "60000",
    "--target-steps", "61000",
    "--save-every", "100",
    "--eval-batches", "16",
    "--log-every", "20",
    "--generation-tokens", "64",
    "--device", "cuda",
    "--clear-stop",
]

worker = subprocess.Popen(command)

try:
    return_code = worker.wait()
except KeyboardInterrupt:
    # Request a save after the current optimizer step.
    REPAIR_RUN.mkdir(parents=True, exist_ok=True)
    (REPAIR_RUN / "STOP").write_text("Requested from Colab\n")
    print("\nStop requested. Waiting for checkpoint save—leave this running.",
          flush=True)
    return_code = worker.wait()

if return_code != 0:
    raise RuntimeError(
        f"Continuation exited with code {return_code}. "
        "Read the error above before rerunning."
    )

progress_file = REPAIR_RUN / "progress.json"
if progress_file.is_file():
    progress = json.loads(progress_file.read_text())
    print("\nSAVED STEP:", progress["saved_step"])
    print("STATUS:", progress["status"])
    print("REPAIRED-VALIDATION BEST BPT:", progress["best_validation_bpt"])
    print("RECOVERY CHECKPOINT:", progress["latest"])


Starting/resuming the repaired-data run to step 61,000.
Checkpoints: /content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1/run_60k_to_61k

SAVED STEP: 61000
STATUS: complete
REPAIRED-VALIDATION BEST BPT: 6.09002
RECOVERY CHECKPOINT: /content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1/run_60k_to_61k/latest.pt


In [13]:
import json
from pathlib import Path

run = Path(
    "/content/drive/MyDrive/Quantonium/PhiState-L4/"
    "whitespace_repair_v1/run_60k_to_61k"
)

migration = json.loads((run / "migration.json").read_text())
progress = json.loads((run / "progress.json").read_text())

baseline = migration["repaired_baseline"]
latest = progress["last_validation"]

print("SAME REPAIRED VALIDATION: 60K → 61K")
for key in ("val_code_bpt", "val_general_bpt", "val_mean_bpt"):
    before, after = baseline[key], latest[key]
    print(f"{key}: {before:.5f} → {after:.5f} | change {after-before:+.5f}")

for filename in ("generation_baseline.json", "generation_step_61000.json"):
    path = run / filename
    if not path.is_file():
        print(f"\nGeneration report not present yet: {filename}")
        continue

    report = json.loads(path.read_text())
    print(f"\n========== STEP {report['step']} ==========")
    for row in report["results"]:
        print(f"\n--- {row['case']} | {row['mode']} ---")
        print(row["text"])
        if row["python_syntax_parses"] is not None:
            print("Python syntax parses:", row["python_syntax_parses"])

SAME REPAIRED VALIDATION: 60K → 61K
val_code_bpt: 9.59986 → 5.26751 | change -4.33235
val_general_bpt: 7.39256 → 6.91252 | change -0.48004
val_mean_bpt: 8.49621 → 6.09002 | change -2.40619

========== STEP 60000 ==========

--- python_add | greedy ---
def add(a, b):
, ^=[j]+1]+1[j]+1]+1[j]+1[j]+1] = [[j]+1]+1[j]+1[j]+1]+1[j]+1[j]+1
Python syntax parses: False

--- python_add | sample ---
def add(a, b):
不不該該該該該該該。 // 止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止止
Python syntax parses: False

--- python_fibonacci | greedy ---
def fibonacci(n):
, .. ^=[j]+1.0.0.1.0.0.0.1.0.2.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0.0
Python syntax parses: False

--- python_fibonacci | sample ---
def fibonacci(n):
同同同同同同同同同据 (43, 250987771925541999946762395466239998, 127} , {x:32.96058956} , {x:32.
Python syntax parses: False

--- ram_vs_storage | greedy ---
User: What is the difference between RAM and storage?
Assistant: utilities: - The first time of the brief

In [ ]:
import subprocess
from pathlib import Path
import json

assert "command" in globals(), "Run the previous setup cell first."
assert Path(command[2]).name == "continue_whitespace_repair.py"

next_command = command.copy()
next_command[next_command.index("--target-steps") + 1] = "65000"

run_dir = Path(
    next_command[next_command.index("--run-dir") + 1]
)
assert (run_dir / "latest.pt").is_file(), "Missing repair checkpoint."

print("Resuming the existing repaired-data run toward step 65,000.",
      flush=True)

worker = subprocess.Popen(next_command)

try:
    return_code = worker.wait()
except KeyboardInterrupt:
    (run_dir / "STOP").write_text("Requested from Colab\n")
    print("Stop requested. Waiting for the checkpoint save.", flush=True)
    return_code = worker.wait()

if return_code != 0:
    raise RuntimeError(f"Training exited with code {return_code}; see output above.")

progress = json.loads((run_dir / "progress.json").read_text())
print("\nSAVED STEP:", progress["saved_step"])
print("STATUS:", progress["status"])
print("BEST REPAIRED VALIDATION BPT:", progress["best_validation_bpt"])
print("CHECKPOINT:", progress["latest"])

Resuming the existing repaired-data run toward step 65,000.
